# Task 74: Validated Heart-Feature Input

Validate numeric input, finite values, coded categories, plausible ranges, and uncommon training-distribution values. Show explicit suggestions and require confirmation before correction.

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

paths=[Path('heart.csv'),Path('Task_74_Validated_Heart_Input/heart.csv'),Path('Heart_Disease_Tasks/Task_74_Validated_Heart_Input/heart.csv')]
data=pd.read_csv(next(p for p in paths if p.is_file())).drop_duplicates().reset_index(drop=True)
features=[c for c in ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'] if c in data]
X,y=data[features].to_numpy(float),data.target.to_numpy(int)
category_names={'sex','cp','fbs','restecg','exang','slope','ca','thal'}
categories={name:sorted(data[name].dropna().astype(int).unique().tolist()) for name in features if name in category_names}
hard_ranges={'age':(18,100),'trestbps':(60,260),'chol':(80,700),'thalach':(40,250),'oldpeak':(-5,15)}
learned={name:(float(data[name].quantile(.01)),float(data[name].quantile(.99))) for name in features if name not in categories}

def validate(values):
    parsed={};issues=[];suggestions={}
    for name in features:
        if name not in values:
            issues.append({'feature':name,'level':'error','message':'Required feature is missing.','suggestion':None});continue
        try:value=float(values[name])
        except (ValueError,TypeError):
            issues.append({'feature':name,'level':'error','message':'Enter a numeric value.','suggestion':None});continue
        if not math.isfinite(value):
            issues.append({'feature':name,'level':'error','message':'NaN or infinity is not allowed.','suggestion':None});continue
        parsed[name]=value
        if name in categories and (not value.is_integer() or int(value) not in categories[name]):
            suggestion=float(min(categories[name],key=lambda item:(abs(item-value),item)));suggestions[name]=suggestion
            issues.append({'feature':name,'level':'error','message':f'Use an observed integer category code: {categories[name]}','suggestion':suggestion});continue
        if name in hard_ranges:
            low,high=hard_ranges[name]
            if value<low or value>high:
                suggestion=float(np.clip(value,low,high));suggestions[name]=suggestion
                issues.append({'feature':name,'level':'error','message':f'Outside configured plausible range [{low}, {high}].','suggestion':suggestion});continue
        if name in learned:
            low,high=learned[name]
            if value<low or value>high:
                suggestion=float(np.clip(value,low,high));suggestions[name]=suggestion
                issues.append({'feature':name,'level':'warning','message':f'Outside training 1st–99th percentile [{low:.3g}, {high:.3g}].','suggestion':suggestion})
    return parsed,issues,suggestions


def logistic_fit(x,labels):
    weights=np.zeros(x.shape[1]);bias=0.
    for _ in range(1200):
        p=1/(1+np.exp(-np.clip(x@weights+bias,-35,35)));error=p-labels
        weights-=.06*(x.T@error/len(labels)+.02*weights);bias-=.06*error.mean()
    return weights,bias

means,scales=X.mean(0),X.std(0);scales[scales==0]=1
scaled=(X-means)/scales
model_weights,model_bias=logistic_fit(scaled,y)

# Demonstrate an invalid row, preview suggestions, and request explicit acceptance.
row={name:str(int(data[name].mode().iloc[0])) if name in categories else str(round(float(data[name].median()),3)) for name in features}
row['age']='125'
if 'cp' in categories:row['cp']='9'
parsed,issues,suggestions=validate(row)
print('Validation issues:');display(pd.DataFrame(issues))
print('Suggested temporary corrections:',suggestions)

# Simulate the user accepting corrections; original row remains unchanged.
corrected=parsed.copy();corrected.update(suggestions)
corrected,remaining,_=validate(corrected)
blocking=[issue for issue in remaining if issue['level']=='error']
assert not blocking,blocking
x=np.array([corrected[name] for name in features]);x=(x-means)/scales
probability=1/(1+np.exp(-np.clip(x@model_weights+model_bias,-35,35)))
print('Original age/code:',row.get('age'),row.get('cp'),'corrected temporary age/code:',corrected.get('age'),corrected.get('cp'))
print('Prediction produced only after explicit correction:',round(float(probability),3))
print('Input dictionary was not mutated:',row.get('age')=='125')

Validation issues:


,feature,level,message,suggestion
0,age,error,"Outside configured plausible range [18, 100].",100.0
1,cp,error,"Use an observed integer category code: [0, 1, ...",3.0


Suggested temporary corrections: {'age': 100.0, 'cp': 3.0}
Original age/code: 125 9 corrected temporary age/code: 100.0 3.0
Prediction produced only after explicit correction: 0.921
Input dictionary was not mutated: True
